# 03: Tracked baselines (MLflow)

Re-runs the three baselines from notebook 02 and logs every run to MLflow (SQLite store on Drive, ADR-001): parameters, metrics, dataset hash, code version, library versions, and the full preprocessing + model pipeline.

**Check at the end:** the metrics must match EXP-001 (same seed, same data). That proves the runs are reproducible.

**Rule:** only one Colab session should write to the MLflow store at a time.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/<your-username>/ai-experiment-lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
from pathlib import Path
import pandas as pd

from ai_experiment_lab.data import load_csv, validate_dataframe, validation_thresholds
from ai_experiment_lab.preprocessing import split_data
from ai_experiment_lab.training import run_baselines
from ai_experiment_lab.tracking import (
    backup_tracking_db, compare_runs, load_pipeline, log_baseline_results, setup_tracking,
)
from ai_experiment_lab.utils.config import get_data_dir, load_config

cfg = load_config(f"{REPO_DIR}/configs/default.yaml")
models_cfg = load_config(f"{REPO_DIR}/configs/model_configs.yaml")["classification"]
d, sp, seed = cfg["dataset"], cfg["split"], cfg["project"]["random_seed"]

df, info = load_csv(Path(get_data_dir()) / d["file"])
result = validate_dataframe(df, d["target"], d["task"], validation_thresholds(cfg), d["id_columns"])
print(result.report())
result.raise_if_rejected()

split = split_data(df, d["target"], d["positive_label"], d["id_columns"],
                   sp["test_size"], seed, sp["stratify"])
print("train:", split.X_train.shape, " test:", split.X_test.shape)

In [ ]:
CV_FOLDS = 5
res = run_baselines(split, models_cfg, cfg["preprocessing"], seed=seed, cv_folds=CV_FOLDS,
                    primary_metric=cfg["selection"]["primary_metric"])
print(res.table[["model", "cv_f1", "test_f1", "test_recall", "test_roc_auc", "fit_seconds"]].round(4).to_string(index=False))

In [ ]:
# Log to MLflow: one run per model.
exp_id = setup_tracking(cfg["tracking"]["experiment_name"])   # URI comes from MLFLOW_TRACKING_URI
run_ids = log_baseline_results(res, split, info, cfg, cv_folds=CV_FOLDS,
                               extra_tags={"experiment.id": "EXP-002", "notebook": "03_tracked_baselines"})
print("experiment id:", exp_id)
for name, rid in run_ids.items():
    print(f"{name:22s} run_id={rid}")

In [ ]:
# Everything in the store for this experiment, best first.
pd.set_option("display.width", 220)
table = compare_runs(cfg["tracking"]["experiment_name"], order_by="cv_f1")
table[["model", "cv_f1", "cv_f1_std", "test_f1", "test_recall", "test_roc_auc", "test_pr_auc",
       "fit_seconds", "code_version", "dataset_sha256"]].round(4)

In [ ]:
# Reproducibility check against EXP-001 (Colab run of notebook 02).
EXP001 = {  # model: (cv_f1, test_f1)
    "logistic_regression": (0.6279, 0.6136),
    "xgboost": (0.6115, 0.6119),
    "random_forest": (0.5395, 0.5305),
}
ok = True
for _, r in res.table.iterrows():
    cv_ref, test_ref = EXP001[r["model"]]
    d_cv, d_test = abs(r["cv_f1"] - cv_ref), abs(r["test_f1"] - test_ref)
    status = "MATCH" if max(d_cv, d_test) < 5e-4 else "DIFFERS"
    ok &= status == "MATCH"
    print(f"{r['model']:22s} cv_f1 diff {d_cv:.4f}  test_f1 diff {d_test:.4f}  -> {status}")
print("REPRODUCED EXP-001" if ok else "Differs from EXP-001: check the lib.* tags (library versions) of both runs.")

In [ ]:
# Load the best run's pipeline back from MLflow and score a raw customer row, the way the API will.
best = table.iloc[0]
pipe = load_pipeline(best["run_id"])
row = split.X_test.iloc[[0]]
print("model:", best["model"], "| run:", best["run_id"][:8])
print("P(churn) for one raw row:", round(float(pipe.predict_proba(row)[0, 1]), 4),
      "| actual:", int(split.y_test.iloc[0]))

In [ ]:
# Snapshot the MLflow database (keeps the 3 newest). Do this at the end of every session.
print("backup:", backup_tracking_db())

## Record

Paste the outputs of the table cell, the reproducibility cell (`REPRODUCED EXP-001` or `DIFFERS`) and the backup path. Results go in `docs/experiments/EXP-002-tracked-baselines.md`.